In [3]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    VotingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ==========================
# Load Data
# ==========================

df = pd.read_csv("encounters_v2.csv")

# ==========================
# Target
# ==========================

target = "readmission_30d_flag"

# ==========================
# Drop leakage / IDs
# ==========================

drop_cols = [
    "readmission_30d_flag",
    "patient_id",
    "encounter_id",
    "claim_id",
    "nurse_emp_id",
    "surgeon_emp_id",
    "npi_billing"
]

X = df.drop(columns=drop_cols, errors="ignore")
y = df[target]
print(y.value_counts())
print(y.value_counts(normalize=True))

# ==========================
# Numerical / Categorical
# ==========================

num_cols = X.select_dtypes(
    include=["int64","float64"]
).columns

cat_cols = X.select_dtypes(
    include=["object"]
).columns

# ==========================
# Preprocessing
# ==========================

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

# ==========================
# Train Test Split
# ==========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==========================
# Models
# ==========================

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

et = ExtraTreesClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=5,
    random_state=42,
    eval_metric="logloss"
)

# ==========================
# Voting Ensemble
# ==========================

ensemble = VotingClassifier(
    estimators=[
        ("rf", rf),
        ("et", et),
        ("xgb", xgb)
    ],
    voting="soft"
)

# ==========================
# Pipeline
# ==========================

model = Pipeline([
    ("prep", preprocessor),
    ("model", ensemble)
])


# ==========================
# Train
# ==========================

model.fit(X_train, y_train)

# ==========================
# Predict and probabilities
# ==========================
probs = model.predict_proba(X_test)[:,1]
preds = (probs >= 0.30).astype(int)

# ==========================
# Evaluation
# ==========================
acc = accuracy_score(y_test, preds)

print("Accuracy =", acc)

print("\nClassification Report\n")
print(classification_report(y_test, preds))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, preds))

readmission_30d_flag
0    2510
1     476
Name: count, dtype: int64
readmission_30d_flag
0    0.840589
1    0.159411
Name: proportion, dtype: float64
Accuracy = 0.7809364548494984

Classification Report

              precision    recall  f1-score   support

           0       0.84      0.91      0.87       503
           1       0.19      0.12      0.14        95

    accuracy                           0.78       598
   macro avg       0.52      0.51      0.51       598
weighted avg       0.74      0.78      0.76       598


Confusion Matrix

[[456  47]
 [ 84  11]]
